
Core Idea

You extract features from your graph, then feed them into logistic regression. The graph gives you relational signal; logistic regression keeps things interpretable (coefficients = feature importance, probabilities = fraud scores).

Step 1 — Extract Graph Features per Node

These become your feature matrix X:

Structural features (topology):

Degree (in/out for directed graphs)
Clustering coefficient — how tightly connected are a node's neighbors?
PageRank / HITS score
Betweenness / closeness centrality

Neighborhood features (guilt by association — key for fraud):

Fraction of neighbors already flagged as fraudulent
Average transaction amount with neighbors
Neighbor degree statistics (mean, max)

Ego-graph features (local subgraph):

Number of triangles the node participates in
Density of the 1-hop neighborhood

In [ ]:
# Step 2 — Build Logistic Regression
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)  # LR needs this

model = LogisticRegression(class_weight='balanced',  # fraud is rare!
                           C=0.1,                    # regularization
                           solver='lbfgs')
model.fit(X_scaled, y)

In [ ]:
# Step 3 — Interpret the Model
import pandas as pd
coef_df = pd.DataFrame({
    'feature': feature_names,
    'coefficient': model.coef_[0]
}).sort_values('coefficient', ascending=False)
# Positive coef → increases fraud probability
# Negative coef → decreases it

Public dataset: Amazon Review Fraud (YelpChi / Amazon) — co-review graphs where nodes are users/products. Hosted by DGL (Deep Graph Library), one line to download.

In [ ]:
import dgl
from dgl.data import FraudAmazonDataset

dataset = FraudAmazonDataset()
graph = dataset[0]

# Node features and labels are already attached
features = graph.ndata['feature']   # shape [N, 25]
labels   = graph.ndata['label']     # 0 = legit, 1 = fraud